# Dataset pazienti: anagrafica e follow-up
Carica i CSV in `Datasets/Processed` e li unisce per ID paziente. Le colonne delle visite sono prefissate con T1–T4; il merge conserva i pazienti dell'anagrafica.

In [15]:
from pathlib import Path
import re
import pandas as pd

def find_project_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if (folder / 'Datasets' / 'Processed').is_dir():
            return folder
    raise FileNotFoundError('Cartella Datasets/Processed non trovata dalla directory corrente')

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'Datasets' / 'Processed'
files = sorted(PROCESSED_DIR.glob('*.csv'))
if not files:
    raise FileNotFoundError(f'Nessun CSV in {PROCESSED_DIR}')
print(f'{len(files)} CSV trovati in {PROCESSED_DIR}')


5 CSV trovati in c:\Users\lucac\Documents\UNI\2_MAGISTRALE\ANNO_2\Semestre_1\Applied Data Science Project\Project\2026-P8-LINKS_ADSP-Dupilumab_Eosinophilia_Early_Prediction\Datasets\Processed


In [16]:
def load_processed_csv(path):
    # La prima riga contiene i nomi principali; la seconda contiene sottoetichette/unità.
    raw = pd.read_csv(path, header=None, dtype='string', encoding='utf-8-sig', low_memory=False)
    if len(raw) < 3:
        raise ValueError(f'CSV senza righe dati: {path.name}')
    main, detail = raw.iloc[0], raw.iloc[1]
    names, counts = [], {}
    for i, (top, sub) in enumerate(zip(main, detail)):
        top = '' if pd.isna(top) else str(top).strip()
        sub = '' if pd.isna(sub) else str(sub).strip()
        base = 'ID paziente' if i == 0 else (f'{top} - {sub}' if top and sub else top or sub or f'colonna_{i + 1}')
        counts[base] = counts.get(base, 0) + 1
        names.append(base if counts[base] == 1 else f'{base}_{counts[base]}')
    data = raw.iloc[2:].copy().reset_index(drop=True)
    data.columns = names
    data['ID paziente'] = data['ID paziente'].str.strip()
    data = data.loc[data['ID paziente'].notna() & data['ID paziente'].ne('')].copy()
    if data['ID paziente'].duplicated().any():
        raise ValueError(f'ID paziente duplicati in {path.name}')
    return data

datasets = {}
for path in files:
    if 'Anagrafica' in path.stem:
        label = 'Anagrafica'
    else:
        match = re.search(r'\bT[1-4]\b', path.stem)
        if not match:
            raise ValueError(f'Nome file non riconosciuto: {path.name}')
        label = match.group()
    if label in datasets:
        raise ValueError(f'Dataset {label} presente più volte')
    datasets[label] = load_processed_csv(path)
    print(f'{label}: {len(datasets[label])} pazienti, {len(datasets[label].columns)} colonne')

if 'Anagrafica' not in datasets:
    raise ValueError('Dataset Anagrafica mancante')


Anagrafica: 112 pazienti, 58 colonne
T1: 112 pazienti, 65 colonne
T2: 112 pazienti, 65 colonne
T3: 112 pazienti, 65 colonne
T4: 112 pazienti, 65 colonne


In [17]:
merged = datasets['Anagrafica'].copy()
for label in ('T1', 'T2', 'T3', 'T4'):
    if label not in datasets:
        continue
    visit = datasets[label].rename(columns=lambda col: col if col == 'ID paziente' else f'{label}_{col}')
    merged = merged.merge(visit, on='ID paziente', how='left', validate='one_to_one')
    missing = (~merged['ID paziente'].isin(datasets[label]['ID paziente'])).sum()
    print(f'{label}: {missing} pazienti dell’anagrafica senza visita')

print(f'Dataset unito: {merged.shape[0]} pazienti × {merged.shape[1]} colonne')
display(merged.columns)


T1: 0 pazienti dell’anagrafica senza visita
T2: 0 pazienti dell’anagrafica senza visita
T3: 0 pazienti dell’anagrafica senza visita
T4: 0 pazienti dell’anagrafica senza visita
Dataset unito: 112 pazienti × 314 colonne


Index(['ID paziente', 'Sesso', 'Età', 'Peso (kg)', 'Altezza (cm)', 'BMI',
       'Biologico precedente (0 no, 1 sì)', 'Categoria Professione',
       'Comorbidità', 'Num Comorbidità',
       ...
       'T4_Eosinofili', 'T4_Cellule metacromatiche',
       'T4_Degranulazione eosinofilo /eritrocitaria', 'T4_Linfociti',
       'T4_Batteri', 'T4_Spore', 'T4_T (soglia)', 'T4_D (discriminazione)',
       'T4_I (identificazione)', 'T4_TDI'],
      dtype='str', length=314)

In [19]:
# Primo superamento di 0,8 ×10³/µL per paziente (Anagrafica, poi T1–T4).
soglia = 0.8
risultati = []
for label in ('Anagrafica', 'T1', 'T2', 'T3', 'T4'):
    if label not in datasets:
        continue
    dati = datasets[label]
    colonna = 'Eosinofili - ×10³/µL'
    if colonna not in dati.columns:
        raise KeyError(f'{colonna} non trovata in {label}')
    valori = pd.to_numeric(dati[colonna].str.replace(',', '.', regex=False), errors='coerce')
    superamenti = dati.loc[valori.gt(soglia), ['ID paziente']].copy()
    superamenti['Eosinofili ×10³/µL'] = valori.loc[superamenti.index]
    superamenti['Primo T'] = 'T0 (Anagrafica)' if label == 'Anagrafica' else label
    risultati.append(superamenti)

primo_superamento = (pd.concat(risultati, ignore_index=True)
                      .drop_duplicates(subset='ID paziente', keep='first'))
print(f'Pazienti con eosinofili > {soglia} ×10³/µL: {len(primo_superamento)}')
print(primo_superamento.to_string(index=False))

# Target binario nel dataset unito: 1 se il paziente supera la soglia almeno una volta.
merged['TARGET'] = merged['ID paziente'].isin(primo_superamento['ID paziente']).astype('int8')
print('\nDistribuzione TARGET:')
print(merged['TARGET'].value_counts().sort_index().to_string())
datasets['Anagrafica']['TARGET'] = datasets['Anagrafica']['ID paziente'].isin(
    primo_superamento['ID paziente']
).astype('int8')
print('\nTARGET nell’Anagrafica:')
anagrafica_target_ordinata = datasets['Anagrafica'][['ID paziente', 'TARGET']].sort_values(
    'ID paziente', key=lambda ids: pd.to_numeric(ids, errors='raise')
)
print(anagrafica_target_ordinata.to_string(index=False))

# Salva il target anche nel CSV originale, mantenendo le due righe di intestazione.
import csv
import os
from tempfile import NamedTemporaryFile

anagrafica_path = next(path for path in files if 'Anagrafica' in path.stem)
target_per_id = dict(zip(datasets['Anagrafica']['ID paziente'], datasets['Anagrafica']['TARGET']))
with anagrafica_path.open('r', encoding='utf-8-sig', newline='') as source:
    righe = list(csv.reader(source))
id_idx = righe[0].index('ID paziente')
target_idx = righe[0].index('TARGET')
for riga in righe[2:]:
    paziente_id = riga[id_idx].strip()
    if paziente_id:
        riga[target_idx] = str(target_per_id[paziente_id])
with NamedTemporaryFile('w', encoding='utf-8-sig', newline='', dir=anagrafica_path.parent, delete=False) as temp:
    csv.writer(temp).writerows(righe)
    temp_path = Path(temp.name)
os.replace(temp_path, anagrafica_path)
print(f'TARGET salvato in {anagrafica_path}')


Pazienti con eosinofili > 0.8 ×10³/µL: 50
ID paziente  Eosinofili ×10³/µL         Primo T
          8                0.82 T0 (Anagrafica)
         19                0.86 T0 (Anagrafica)
         21                1.17 T0 (Anagrafica)
         43                1.28 T0 (Anagrafica)
         52                 1.2 T0 (Anagrafica)
         58                 0.9 T0 (Anagrafica)
         62                 6.5 T0 (Anagrafica)
         83                0.88 T0 (Anagrafica)
          7                 0.9              T1
         31                1.06              T1
         57                3.12              T1
         89                0.99              T1
          9                 2.5              T2
         10                1.22              T2
         11                 1.5              T2
         13                0.98              T2
         23                1.01              T2
         27                0.92              T2
         34                0.92              T